# Project Design

## Research Question

Did NWS staffing cuts in 2025 result in statistically significant changes in warning performance?

## Data Source

**COW API** (`mesonet.agron.iastate.edu/api/1/cow.json`)
- 122 WFOs × 6 complete calendar years (2020–2025)
- Filtered to phenomena: **TO** (Tornado), **SV** (Severe Thunderstorm), **FF** (Flash Flood)
- Collected May 2026; 2025 is a full year with no partial-year bias
- Year boundary verified: no duplicate or misassigned events across year files

## Data Storage

```
data/
├── wfo_list.csv
├── raw/
│   └── COW/
│       └── {WFO}_{YEAR}.json   # one file per WFO-year, immutable
└── processed/                  # analysis-ready derived files
```

## Data Model

Three tables extracted from the raw JSON:

| Table | Grain | Key fields |
|---|---|---|
| `events` | One row per warning event | `id`, `wfo`, `year`, `phenomena`, `fcster`, `issue`, `expire`, `verify`, `lead0` |
| `stormreports` | One row per LSR | `wfo`, `year`, `valid`, `lsrtype`, `warned`, `leadtime`, `events` (FK to events.id) |
| *(fcster analysis)* | TBD — unique forecasters per WFO per year | Derived from `events.fcster` |

## Analysis Design

- **Pre/post split:** 2020–2024 baseline, 2025 treatment
- **Primary metrics:** POD, FAR, CSI, avg lead time — derived from event-level data
- **Staffing proxy:** Year-over-year change in unique `fcster` values within each WFO

## Methodological Notes

1. **LSR underreporting:** LSRs are filed voluntarily; miss rates should be interpreted as "among reported events." A drop in unwarned reports in 2025 could reflect fewer LSRs filed, not better performance.
2. **`fcster` field quality:** Format varies by WFO (last names, initials, badge numbers). Cross-WFO headcount comparisons are invalid. Within-WFO year-over-year changes may be valid — pending verification of within-WFO format consistency.
3. **Magnitude field:** Not comparable across phenomena (EF scale for TO, mph for SV, inches for FF). Must split by `lsrtype` before any magnitude analysis.
4. **Small-sample WFOs:** WFOs with very few events in a year will produce unreliable per-WFO statistics. A minimum events threshold may be needed.
5. **Non-CONUS offices:** Alaska, Hawaii, Guam, Puerto Rico have fundamentally different weather patterns. Consider flagging or excluding from the main analysis.

# Data Collection

In [1]:
import logging
import sys
from pathlib import Path

sys.path.append("../src")

from collection import COWClient, COWCollector, WFORegistry

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)

# --- Configuration ---

# Paths
WFO_FILE = Path("../data/wfo_list.csv")
COW_DIR  = Path("../data/raw/COW")

# IEM COW API endpoint
COW_BASE_URL = "https://mesonet.agron.iastate.edu/api/1/cow.json"

# VTEC phenomena codes to collect and analyze:
#   TO = Tornado Warning
#   SV = Severe Thunderstorm Warning
#   FF = Flash Flood Warning
PHENOMENA = ["TO", "SV", "FF"]

# Analysis window: 2020–2024 are the pre-treatment baseline years;
# 2025 is the treatment year (post-staffing-cuts).
YEARS = list(range(2020, 2026))

# Seconds to pause between API requests to avoid overwhelming the IEM server
RATE_LIMIT = 0.3

In [2]:
registry  = WFORegistry(WFO_FILE)
client    = COWClient(phenomena=PHENOMENA, base_url=COW_BASE_URL)
collector = COWCollector(registry, client, COW_DIR, years=YEARS, rate_limit=RATE_LIMIT)

print(registry)
print(client)
print(collector)

summary = collector.collect()
print(summary)

14:51:06  INFO     Collection complete — {'wrote': 0, 'skipped': 732, 'failed': 0}


WFORegistry(122 offices from wfo_list.csv)
COWClient(phenomena=['TO', 'SV', 'FF'], url=https://mesonet.agron.iastate.edu/api/1/cow.json)
COWCollector(122 WFOs, years=2020-2025, output=../data/raw/COW)
{'wrote': 0, 'skipped': 732, 'failed': 0}
